# Four-model test on a Colab GPU: AEROBLADE, TruFor, ForMa, PSCC-Net

Runs the group's 4-model test (`model_test/`, branch `model-test-gpu`) on Colab's GPU, with **ForMa's compiled
`selective_scan` CUDA kernel** (the fast path it is designed for, which cannot be built on Windows). All four models
are re-measured on the same GPU so their timings are comparable with each other.

**Before you start:** `Runtime > Change runtime type > T4 GPU` (or any GPU), then `Runtime > Run all`.

| step | time on a T4 |
|---|---|
| Python 3.10 env with the same versions as the laptop (PyTorch 2.1.2 + CUDA 12.1) | ~5 min |
| code and weights (TruFor 261 MB, ForMa 443 MB, PSCC-Net) | ~3 min |
| build ForMa's kernel, check it against the PyTorch version | ~10-20 min |
| run the 4 models on 65 images + report | ~15-25 min |

At the end a zip with `REPORT.md`, figures, per-image CSVs and profiles is downloaded. Colab disconnects idle
sessions, so keep the tab open. Re-running a cell is safe: finished steps are skipped.

Different GPU from the laptop run: compare models **within** this notebook's report, not against the laptop numbers.

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv
import shutil
assert shutil.which("nvidia-smi"), "No GPU: Runtime > Change runtime type > T4 GPU, then run again"

## 2. Code (branch `model-test-gpu`, includes the 65 test images)

In [ ]:
%cd /content
![ -d /content/g8 ] || git clone -q --depth 1 -b model-test-gpu https://github.com/Wissemamr/Group8_Benchmarking_inpainting_detection_with_small_models /content/g8
%cd /content/g8
!git log --oneline -1

## 3. Python 3.10 environment, same versions as the laptop's `Aeroblade` env
Colab's own Python is left untouched (it is 3.12, too new for PyTorch 2.1.2).

In [ ]:
%%bash
set -e
cd /content
if [ ! -x /content/mt/bin/python ]; then
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba
  /content/bin/micromamba create -y -q -p /content/mt -c conda-forge python=3.10
fi
P=/content/mt/bin/python
$P -m pip install -q --upgrade pip
$P -m pip install -q -r /content/g8/AEROBLADE/requirements-gpu.txt
$P -m pip install -q "timm==0.5.4" "yacs==0.1.8" "fvcore==0.1.5.post20221221" \
    "opencv-python-headless==4.10.0.84" "einops==0.8.0" psutil ninja packaging     "setuptools==69.5.1" "wheel==0.44.0"   # PyTorch 2.1's extension builder needs pkg_resources
$P -c "import torch; print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU', torch.cuda.get_device_name(0))"

## 4. Third-party code and weights (TruFor official code + weights, ForMa checkpoint, official PSCC-Net)

In [ ]:
!cd /content/g8 && /content/mt/bin/python model_test/setup_models.py

**Only if the ForMa download above failed** (Google Drive sometimes refuses): run this cell and upload your own
`ForMa_weights.pth` (e.g. `D:\ForMa_weights.pth`). It is checked against the authors' checksum.

In [ ]:
import hashlib, os, shutil
target = "/content/g8/ForMa/ForMa_weights/ForMa_weights.pth"
expected = "3c28719a48848411c4647e2460887d9a5d6eb64f2c3e3a77f5061c686925c2ee"
if os.path.exists(target):
    print("ForMa weights already in place:", target)
else:
    from google.colab import files
    up = files.upload()
    os.makedirs(os.path.dirname(target), exist_ok=True)
    shutil.move(next(iter(up)), target)
h = hashlib.sha256(open(target, "rb").read()).hexdigest()
print("SHA-256 OK" if h == expected else f"WRONG FILE: sha256 {h}")

## 5. Build ForMa's `selective_scan` CUDA kernel
VMamba's `kernels/selective_scan` (= "selective_scan 0.0.2" in ForMa's README), commit `bc2453dc568a`, built only for
this GPU, with the toolchain that matches PyTorch 2.1.2: the **CUDA 12.1** compiler (installed into the env, so Colab's own
CUDA version does not matter), GCC 12 or older, one compile job (stays within Colab's memory). This recipe was tested.

In [ ]:
import re, shutil, subprocess
mm, env = "/content/bin/micromamba", "/content/mt"
if not shutil.os.path.exists(f"{env}/bin/nvcc"):
    print("installing the CUDA 12.1 compiler and headers into the env (~1 GB)...")
    subprocess.run(f"{mm} install -y -q -p {env} -c nvidia/label/cuda-12.1.1 --override-channels "
                   "cuda-nvcc cuda-cudart-dev cuda-cccl libcublas-dev libcusparse-dev libcusolver-dev libcurand-dev",
                   shell=True, check=True)
nvcc = subprocess.run([f"{env}/bin/nvcc", "--version"], capture_output=True, text=True).stdout.strip().splitlines()[-1]
# CUDA 12.1 accepts GCC up to 12: use the default gcc if it is old enough, else install g++-12
gcc_major = int(subprocess.run(["gcc", "-dumpversion"], capture_output=True, text=True).stdout.split(".")[0])
host = ""
if gcc_major > 12:
    subprocess.run("apt-get -qq update && apt-get -qq install -y gcc-12 g++-12", shell=True, check=True)
    host = "export CC=gcc-12 CXX=g++-12 NVCC_PREPEND_FLAGS='-ccbin /usr/bin/g++-12'\n"
cap = subprocess.run([f"{env}/bin/python", "-c", "import torch; print('%d.%d' % torch.cuda.get_device_capability(0))"],
                     capture_output=True, text=True).stdout.strip()
open("/content/cuda_env.sh", "w").write(
    f"export CUDA_HOME={env}\nexport PATH={env}/bin:$PATH\nexport TORCH_CUDA_ARCH_LIST={cap}\nexport MAX_JOBS=1\n" + host)
print("nvcc:", nvcc, "| gcc", gcc_major, "(g++-12 used)" if host else "", "| GPU compute capability", cap)

In [ ]:
%%bash
set -eo pipefail
source /content/cuda_env.sh
cd /content
if /content/mt/bin/python -c "import torch, selective_scan_cuda_oflex" 2>/dev/null; then
  echo "kernel already built"; exit 0
fi
# PyTorch 2.1's extension builder imports pkg_resources: needs setuptools < 70 and a matching wheel
/content/mt/bin/python -m pip install -q "setuptools==69.5.1" "wheel==0.44.0" ninja packaging
[ -d VMamba ] || git clone -q --filter=blob:none https://github.com/MzeroMiko/VMamba
git -C VMamba checkout -q bc2453dc568a
cd VMamba/kernels/selective_scan
git checkout -q -- setup.py && rm -rf build          # clean state after a failed attempt
sed -i 's/"--threads", "4"/"--threads", "1"/' setup.py  # one nvcc thread: keeps memory low
echo "building (about 5-15 min, full log in /content/kernel_build.log)..."
if ! /content/mt/bin/python -m pip install . --no-build-isolation -v > /content/kernel_build.log 2>&1; then
  echo "BUILD FAILED. First error lines:"
  grep -n -E "error|Killed|fatal" /content/kernel_build.log | grep -v -E "Werror|error_|_error|errors\\.h|Wno-error" | head -15
  echo "--- last 15 lines:"; tail -n 15 /content/kernel_build.log; exit 1
fi
/content/mt/bin/python -c "import torch, selective_scan_cuda_oflex as k; print('kernel OK:', k.__file__)"

Correctness check: the compiled kernel against ForMa's own PyTorch implementation on random inputs.

In [ ]:
%%bash
cat > /content/check_kernel.py <<'EOF'
import sys, torch
sys.path.insert(0, "/content/g8/ForMa")
from models.csms6s import selective_scan_fn, WITH_SELECTIVESCAN_OFLEX
assert WITH_SELECTIVESCAN_OFLEX, "ForMa does not see the kernel"
torch.manual_seed(0)
Bt, K, Cd, N, L = 1, 4, 32, 4, 1024
KC = K * Cd
dev = "cuda"
u, delta = torch.randn(Bt, KC, L, device=dev), torch.randn(Bt, KC, L, device=dev)
A = -torch.rand(KC, N, device=dev)
B, C = torch.randn(Bt, K, N, L, device=dev), torch.randn(Bt, K, N, L, device=dev)
D, bias = torch.randn(KC, device=dev), torch.randn(KC, device=dev)
yk = selective_scan_fn(u, delta, A, B, C, D, bias, True, True, backend="oflex")
yt = selective_scan_fn(u, delta, A, B, C, D, bias, True, True, backend="torch")
rel = ((yk - yt).abs().max() / yt.abs().max()).item()
print(f"kernel vs PyTorch: max relative difference {rel:.2e}")
assert rel < 1e-3, "kernel output differs from the PyTorch implementation"
print("kernel matches ForMa's PyTorch selective scan")
EOF
/content/mt/bin/python /content/check_kernel.py

## 6. Run the models (65 images each, batch 1, after 3 warm-up images)
ForMa first. Each cell writes `model_test/outputs_colab/<model>/`; re-running a cell redoes that model only.

**ForMa, now with the compiled kernel (look for `selective_scan_oflex` in the report notes)**

In [ ]:
!cd /content/g8 && PYTHONWARNINGS=ignore /content/mt/bin/python model_test/run_model.py --model forma --out model_test/outputs_colab/forma

**AEROBLADE (first run downloads the SD2 autoencoder and VGG16)**

In [ ]:
!cd /content/g8 && PYTHONWARNINGS=ignore /content/mt/bin/python model_test/run_model.py --model aeroblade --out model_test/outputs_colab/aeroblade

**TruFor**

In [ ]:
!cd /content/g8 && PYTHONWARNINGS=ignore /content/mt/bin/python model_test/run_model.py --model trufor --out model_test/outputs_colab/trufor

**PSCC-Net (official code and checkpoints)**

In [ ]:
!cd /content/g8 && PYTHONWARNINGS=ignore /content/mt/bin/python model_test/run_model.py --model pscc --out model_test/outputs_colab/pscc

## 7. Report

In [ ]:
!cd /content/g8 && /content/mt/bin/python model_test/make_report.py --outputs model_test/outputs_colab
from IPython.display import Markdown, Image, display
rep = "/content/g8/model_test/outputs_colab/report/"
display(Markdown(open(rep + "REPORT.md", encoding="utf-8").read().split("## Figures")[0]))
for f in ("latency.png", "memory.png", "quality_vs_speed.png", "examples.png"):
    try:
        display(Image(rep + f))
    except Exception as e:
        print(f, "missing:", e)

## 8. Download the results

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive("/content/model_test_colab_results", "zip", "/content/g8/model_test", "outputs_colab")
files.download(zip_path)

Optional: keep a copy in your Google Drive instead (asks for permission).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp /content/model_test_colab_results.zip /content/drive/MyDrive/
!ls -la /content/drive/MyDrive/model_test_colab_results.zip